# Gold Layer

## Claims Aggregations

### Daily Claims Aggregation

Creates daily business-level metrics from silver claims data.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
silver_claims = spark.table("silver_claims")

In [0]:
silver_claims.printSchema()

In [0]:
daily_claims = (
    silver_claims
    .withColumn(
        "claim_date",
        F.to_date("claim_datetime")
    )
    .groupBy("claim_date")
    .agg(
        F.count("*").alias("number_of_claims"),
        F.sum("total_claim_amount").alias("total_claim_amount"),
        F.sum("injury_claim_amount").alias("injury_claim_amount"),
        F.sum("property_claim_amount").alias("property_claim_amount"),
        F.sum("vehicle_claim_amount").alias("vehicle_claim_amount"),
        F.round(
            F.avg("incident_hour")
        ).alias("average_incident_hour"),
        F.round(
            F.avg("driver_age")
        ).alias("average_driver_age")
    )
)

In [0]:


w = Window.orderBy("claim_date")

In [0]:
daily_claims = (
    daily_claims
    .withColumn(
        "pct_change_number_of_claims",
        F.round(
            (
                (F.col("number_of_claims") -
                 F.lag("number_of_claims").over(w))
                /
                F.lag("number_of_claims").over(w)
            ) * 100,
            2
        )
    )
    .withColumn(
        "30d_rolling_avg_total_claim_amount",
        F.round(
            F.avg("total_claim_amount")
            .over(w.rowsBetween(-30,0))
        )
    )
)

## Weekly Claims Aggregation

Creates weekly-level business metrics from silver claims data.

In [0]:
weekly_claims = (
    silver_claims
    .withColumn(
        "claim_year_week",
        F.concat_ws(
            "-",
            F.year("claim_datetime"),
            F.lpad(
                F.weekofyear("claim_datetime"),
                2,
                "0"
            )
        )
    )
)

In [0]:
weekly_claims = (
    weekly_claims
    .groupBy("claim_year_week")
    .agg(
        F.count("*").alias("number_of_claims"),

        F.sum("total_claim_amount")
        .alias("total_claim_amount"),

        F.sum("injury_claim_amount")
        .alias("injury_claim_amount"),

        F.sum("property_claim_amount")
        .alias("property_claim_amount"),

        F.sum("vehicle_claim_amount")
        .alias("vehicle_claim_amount"),

        F.round(
            F.avg("incident_hour")
        ).alias("average_incident_hour"),

        F.round(
            F.avg("driver_age")
        ).alias("average_driver_age")
    )
)

In [0]:
w_week = Window.orderBy("claim_year_week")

In [0]:
weekly_claims = (
    weekly_claims
    .withColumn(
        "pct_change_number_of_claims",
        F.round(
            (
                (
                    F.col("number_of_claims")
                    -
                    F.lag("number_of_claims").over(w_week)
                )
                /
                F.lag("number_of_claims").over(w_week)
            ) * 100,
            2
        )
    )
    .withColumn(
        "3m_rolling_avg_total_claim_amount",
        F.round(
            F.avg("total_claim_amount")
            .over(
                w_week.rowsBetween(-3,0)
            )
        )
    )
)

In [0]:
weekly_claims.show(10)

In [0]:
weekly_claims.printSchema()

In [0]:
(
    weekly_claims.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_claims_weekly")
)

## Monthly Claims Aggregation

Creates monthly-level business metrics from silver claims data.

In [0]:
monthly_claims = (
    silver_claims
    .withColumn(
        "claim_year_month",
        F.concat_ws(
            "-",
            F.year("claim_datetime"),
            F.lpad(
                F.month("claim_datetime"),
                2,
                "0"
            )
        )
    )
)

In [0]:
monthly_claims = (
    monthly_claims
    .groupBy("claim_year_month")
    .agg(
        F.count("*")
        .alias("number_of_claims"),

        F.sum("total_claim_amount")
        .alias("total_claim_amount"),

        F.sum("injury_claim_amount")
        .alias("injury_claim_amount"),

        F.sum("property_claim_amount")
        .alias("property_claim_amount"),

        F.sum("vehicle_claim_amount")
        .alias("vehicle_claim_amount"),

        F.round(
            F.avg("incident_hour")
        ).alias("average_incident_hour"),

        F.round(
            F.avg("driver_age")
        ).alias("average_driver_age")
    )
)

In [0]:
w_month = Window.orderBy("claim_year_month")

In [0]:
monthly_claims = (
    monthly_claims
    .withColumn(
        "pct_change_number_of_claims",
        F.round(
            (
                (
                    F.col("number_of_claims")
                    -
                    F.lag("number_of_claims").over(w_month)
                )
                /
                F.lag("number_of_claims").over(w_month)
            ) * 100,
            2
        )
    )
    .withColumn(
        "3m_rolling_avg_total_claim_amount",
        F.round(
            F.avg("total_claim_amount")
            .over(
                w_month.rowsBetween(-3,0)
            )
        )
    )
)

In [0]:
monthly_claims.show(10)

In [0]:
monthly_claims.printSchema()

In [0]:
(
    monthly_claims.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_claims_monthly")
)

# Accident Aggregations

Business-level accident metrics generated from silver accident records.

In [0]:
silver_accidents = spark.table("silver_accidents")

In [0]:
silver_accidents.printSchema()

In [0]:
daily_accidents = (
    silver_accidents
    .groupBy("accident_date")
    .agg(
        F.count("*")
        .alias("number_of_accidents"),

        F.round(
            F.avg("accident_hour")
        )
        .alias("average_accident_hour"),

        F.expr(
            "percentile(number_of_vehicles_involved, array(0.75))[0]"
        )
        .alias("75_pctl_number_of_vehicles_involved"),

        F.max("borough")
        .alias("most_common_borough"),

        F.max("zip_code")
        .alias("most_common_zip_code")
    )
)

In [0]:
w_accident = Window.orderBy("accident_date")

In [0]:
silver_accidents = (
    silver_accidents
    .withColumn(
        "accident_hour",
        F.hour(
            F.try_to_timestamp(
                F.col("accident_time"),
                F.lit("H:mm")
            )
        )
    )
)

In [0]:
daily_accidents = (
    daily_accidents

    .withColumn(
        "pct_change_number_of_accidents",
        F.round(
            (
                (
                    F.col("number_of_accidents")
                    -
                    F.lag("number_of_accidents")
                    .over(w_accident)
                )
                /
                F.lag("number_of_accidents")
                .over(w_accident)
            ) * 100,
            2
        )
    )

    .withColumn(
        "30d_rolling_avg_number_of_accidents",
        F.round(
            F.avg("number_of_accidents")
            .over(
                w_accident.rowsBetween(-30,0)
            )
        )
    )
)

In [0]:
silver_accidents.select(
    "accident_time",
    "accident_hour"
).show(10)

In [0]:
silver_accidents.select(
    "accident_time",
    "accident_hour"
).show(10)

In [0]:
daily_accidents = (
    silver_accidents
    .groupBy("accident_date")
    .agg(
        F.count("*").alias("number_of_accidents"),

        F.round(
            F.avg("accident_hour")
        ).alias("average_accident_hour"),

        F.max("borough")
        .alias("most_common_borough"),

        F.max("zip_code")
        .alias("most_common_zip_code")
    )
)

In [0]:
w_accident = Window.orderBy("accident_date")

In [0]:
(
    daily_accidents.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_accidents_daily")
)

## Weekly Accident Aggregation

Creates weekly-level accident metrics from silver accident records.

In [0]:
weekly_accidents = (
    silver_accidents
    .withColumn(
        "accident_year_week",
        F.concat_ws(
            "-",
            F.year("accident_date"),
            F.lpad(
                F.weekofyear("accident_date"),
                2,
                "0"
            )
        )
    )
)

In [0]:
weekly_accidents = (
    weekly_accidents
    .groupBy("accident_year_week")
    .agg(
        F.count("*")
        .alias("number_of_accidents"),

        F.round(
            F.avg("accident_hour")
        )
        .alias("average_accident_hour"),

        F.max("borough")
        .alias("most_common_borough"),

        F.max("zip_code")
        .alias("most_common_zip_code")
    )
)

In [0]:
w_accident_week = Window.orderBy("accident_year_week")

In [0]:
weekly_accidents = (
    weekly_accidents

    .withColumn(
        "pct_change_number_of_accidents",
        F.round(
            (
                (
                    F.col("number_of_accidents")
                    -
                    F.lag("number_of_accidents")
                    .over(w_accident_week)
                )
                /
                F.lag("number_of_accidents")
                .over(w_accident_week)
            ) * 100,
            2
        )
    )

    .withColumn(
        "3m_rolling_avg_number_of_accidents",
        F.round(
            F.avg("number_of_accidents")
            .over(
                w_accident_week.rowsBetween(-3,0)
            )
        )
    )
)

In [0]:
weekly_accidents.show(10)

In [0]:
weekly_accidents.printSchema()

In [0]:
(
    weekly_accidents.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_accidents_weekly")
)

## Monthly Accident Aggregation

Creates monthly-level accident metrics from silver accident records.

In [0]:
monthly_accidents = (
    silver_accidents
    .withColumn(
        "accident_year_month",
        F.concat_ws(
            "-",
            F.year("accident_date"),
            F.lpad(
                F.month("accident_date"),
                2,
                "0"
            )
        )
    )
)

In [0]:
monthly_accidents = (
    monthly_accidents
    .groupBy("accident_year_month")
    .agg(
        F.count("*")
        .alias("number_of_accidents"),

        F.round(
            F.avg("accident_hour")
        )
        .alias("average_accident_hour"),

        F.max("borough")
        .alias("most_common_borough"),

        F.max("zip_code")
        .alias("most_common_zip_code")
    )
)

In [0]:
w_accident_month = Window.orderBy("accident_year_month")

In [0]:
monthly_accidents = (
    monthly_accidents

    .withColumn(
        "pct_change_number_of_accidents",
        F.round(
            (
                (
                    F.col("number_of_accidents")
                    -
                    F.lag("number_of_accidents")
                    .over(w_accident_month)
                )
                /
                F.lag("number_of_accidents")
                .over(w_accident_month)
            ) * 100,
            2
        )
    )

    .withColumn(
        "3m_rolling_avg_number_of_accidents",
        F.round(
            F.avg("number_of_accidents")
            .over(
                w_accident_month.rowsBetween(-3,0)
            )
        )
    )
)

In [0]:
monthly_accidents.show(10)

In [0]:
monthly_accidents.printSchema()

In [0]:
(
    monthly_accidents.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_accidents_monthly")
)

## Monthly Policy Aggregation

Creates monthly-level policy business metrics from silver policy records.

In [0]:
silver_policies = spark.table("silver_policies")

In [0]:
silver_policies.printSchema()

In [0]:
monthly_policies = (
    silver_policies
    .withColumn(
        "year_month",
        F.concat_ws(
            "-",
            F.year("issue_date"),
            F.lpad(
                F.month("issue_date"),
                2,
                "0"
            )
        )
    )
)

In [0]:
issued_policies = (
    monthly_policies
    .withColumn(
        "issue_age_of_vehicle",
        F.year("issue_date") - F.col("model_year").cast("integer")
    )
    .groupBy("year_month")
    .agg(
        F.count("*")
        .alias("policies_issued"),

        F.round(
            F.sum("sum_insured")
        )
        .alias("exposure"),

        F.round(
            F.avg("issue_age_of_vehicle")
        )
        .alias("avg_issue_age_of_vehicle")
    )
)

In [0]:
expired_policies = (
    silver_policies
    .withColumn(
        "year_month",
        F.concat_ws(
            "-",
            F.year("expiry_date"),
            F.lpad(
                F.month("expiry_date"),
                2,
                "0"
            )
        )
    )
    .groupBy("year_month")
    .agg(
        F.count("*")
        .alias("policies_expired")
    )
)

In [0]:
gold_policies_monthly = (
    issued_policies
    .join(
        expired_policies,
        on="year_month",
        how="left"
    )
    .fillna(
        0,
        [
            "policies_issued",
            "policies_expired"
        ]
    )
    .orderBy("year_month")
)

In [0]:
gold_policies_monthly.show(10)

In [0]:
gold_policies_monthly.printSchema()

In [0]:
(
    gold_policies_monthly.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_policies_monthly")
)